# Reader agreement: exact rating vs suspicion

Emmaus

Do readers agree more on whether a nodule is suspicious than on its exact 1-5 rating?

**Method (one reader vs the rest):**
1. Use only nodules rated by 4 readers.
2. Compare each reader's rating with the median of the other 3 readers' ratings, so each nodule gives 4 comparisons.
3. Count two kinds of agreement:
   - **exact rating:** the reader's rating equals the others' median
   - **suspicion:** both are suspicious (4-5) or both are not suspicious (1-3)

All splits are used: this describes the readers, not a model, so there is nothing to keep unseen.
The same comparison is in `pipeline/6baselines.py` (one reader vs the rest), which reports it per split.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

project_folder = Path(os.getcwd()).parent
nodules_labels_path = project_folder / "data" / "nodules_labels.csv"
df = pd.read_csv(nodules_labels_path)

four = df[df["num_readers"] == 4]
print(f"{len(four)} of {len(df)} nodules have 4 readers")

900 of 1885 nodules have 4 readers


## One reader vs the median of the other three

The median of 3 ratings is the middle one after sorting, so it is always a whole rating (no .5 to round).

In [2]:
own, rest = [], []          # each reader's rating, and the median of the other 3 readers
for ratings in four["ratings"]:
    r = [int(x) for x in ratings.split(";")]
    for i in range(4):
        others = sorted(r[:i] + r[i + 1:])     # the other 3 readers' ratings
        own.append(r[i])
        rest.append(others[1])                 # the middle one = their median
own, rest = np.array(own), np.array(rest)

# e.g. ratings 2;3;4;4, first reader: others 3, 4, 4 -> median 4, so 2 vs 4
print(f"{len(own)} comparisons (4 per nodule)")

3600 comparisons (4 per nodule)


## Agreement

In [3]:
exact = own == rest                             # same exact rating
suspicion = (own >= 4) == (rest >= 4)           # same side of the 3 | 4 line

n = len(own)
print(f"exact rating agrees:      {exact.sum()} of {n} ({exact.mean():.1%})")
print(f"suspicion category agrees: {suspicion.sum()} of {n} ({suspicion.mean():.1%})")

exact rating agrees:      1496 of 3600 (41.6%)
suspicion category agrees: 2560 of 3600 (71.1%)


## Breakdown: how far apart the disagreements are

Distance = |reader's rating - median of the other 3|. A distance of 0 is exact agreement.

In [4]:
distance = np.abs(own - rest)
table = pd.DataFrame({
    "comparisons": pd.Series(distance).value_counts().sort_index(),
})
table["share"] = (table["comparisons"] / n).round(3)
table.index.name = "distance"
table

,comparisons,share
distance,,
0,1496,0.416
1,1472,0.409
2,529,0.147
3,87,0.024
4,16,0.004
